## Overview

`Conv_Decomposer` factorizes Conv2d layers into smaller convolutions using 4 different mathematical decompositions. Each trades off compression, accuracy, and inference overhead differently.

In [1]:
#| include: false
from fastai.vision.all import *
from fasterai.misc.all import *
import torch
import torch.nn as nn

## 1. Setup and Data

In [2]:
path = untar_data(URLs.PETS)
files = get_image_files(path/"images")
def label_func(f): return f[0].isupper()
dls = ImageDataLoaders.from_name_func(path, files, label_func, item_tfms=Resize(64))

## 2. Train the Model

We use ResNet-18 — a Conv-heavy model where Tucker decomposition is effective:

In [3]:
learn = vision_learner(dls, resnet18, metrics=accuracy)
learn.unfreeze()
learn.fit(3)

epoch,train_loss,valid_loss,accuracy,time
0,0.521978,0.406796,0.833559,00:02
1,0.329455,0.378803,0.863329,00:02
2,0.265389,0.282685,0.893775,00:02


## 3. Compare Decomposition Methods

Let's decompose the same model with all 4 methods and compare:

In [10]:
import copy, time

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def measure_latency(model, x, warmup=10, steps=50):
    model.eval()
    with torch.no_grad():
        for _ in range(warmup): model(x)
        if x.is_cuda: torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(steps): model(x)
        if x.is_cuda: torch.cuda.synchronize()
    return (time.perf_counter() - t0) / steps * 1000  # ms

learn.model = learn.model.cpu()

original_params = count_params(learn.model)
device = next(learn.model.parameters()).device
x_bench = torch.randn(8, 3, 64, 64, device=device)
base_ms = measure_latency(learn.model, x_bench)

decomposer = Conv_Decomposer()

print(f"{'Method':<10} {'Layers':>6} {'Params':>10} {'Compress':>9} {'Latency':>9} {'Speedup':>8}")
print("-" * 60)
print(f"{'original':<10} {'—':>6} {original_params:>10,} {'1.0x':>9} {base_ms:>8.2f}ms {'1.0x':>8}")

for method in ['svd', 'spatial', 'tucker', 'cp']:
    model_dec = decomposer.decompose(copy.deepcopy(learn.model), 0.5, method=method)
    params = count_params(model_dec)
    ms = measure_latency(model_dec, x_bench)
    n_layers = {'svd': 2, 'tucker': 3, 'spatial': 2, 'cp': 4}[method]
    print(f"{method:<10} {n_layers:>6} {params:>10,} {original_params/params:>8.1f}x {ms:>8.2f}ms {base_ms/ms:>7.1f}x")

Method     Layers     Params  Compress   Latency  Speedup
------------------------------------------------------------
original        — 11,704,896      1.0x     7.10ms     1.0x
svd             2  6,426,195      1.8x     7.16ms     1.0x
spatial         2  4,388,736      2.7x     7.53ms     0.9x
tucker          3  4,723,619      2.5x     8.81ms     0.8x
cp              4  1,897,873      6.2x     8.66ms     0.8x


### How Each Method Decomposes a Conv2d(64, 128, 3×3)

**SVD** (2 layers) — decomposes output channels:
```
Conv2d(64, R, 3×3)  →  Conv2d(R, 128, 1×1)
```

**Tucker** (3 layers) — decomposes both channel dimensions:
```
Conv2d(64, R_in, 1×1)  →  Conv2d(R_in, R_out, 3×3)  →  Conv2d(R_out, 128, 1×1)
```

**Spatial** (2 layers) — decomposes the kernel spatially:
```
Conv2d(64, 128×R, 3×1)  →  Conv2d(128×R, 128, 1×3, groups=128)
```

**CP** (4 layers) — decomposes channels AND spatial:
```
Conv2d(64, R, 1×1)  →  Conv2d(R, R, 3×1, dw)  →  Conv2d(R, R, 1×3, dw)  →  Conv2d(R, 128, 1×1)
```

Each targets a different source of redundancy. Tucker is the best general-purpose choice; CP gives maximum compression but may need more fine-tuning.

## 4. Accuracy Impact (Before Fine-Tuning)

Each method has a different reconstruction error — let's measure accuracy drop:

In [11]:
baseline = Learner(dls, learn.model, metrics=accuracy).validate()[1]
print(f"{'Method':<10} {'Accuracy':>10} {'vs Baseline':>12}")
print("-" * 35)
print(f"{'original':<10} {baseline*100:>9.1f}% {'':>12}")

for method in ['svd', 'tucker', 'spatial', 'cp']:
    model_dec = decomposer.decompose(copy.deepcopy(learn.model), 0.5, method=method)
    acc = Learner(dls, model_dec, metrics=accuracy).validate()[1]
    print(f"{method:<10} {acc*100:>9.1f}% {(acc-baseline)*100:>+11.1f}%")

Method       Accuracy  vs Baseline
-----------------------------------
original        89.4%             


svd             66.2%       -23.2%


tucker          65.6%       -23.8%


spatial         66.6%       -22.7%


cp              66.6%       -22.7%


Fine-tuning recovers most of the accuracy:

```python
new_learn = Learner(dls, model_dec, metrics=accuracy)
new_learn.fit_one_cycle(3, 1e-4)
```

## 5. Activation-Aware Decomposition

Standard decomposition treats all channels equally. By passing calibration data, channels that are actually used by the data get prioritized — same idea as Wanda for pruning.

In [ ]:
# Get a calibration batch from the training set
cal_data = [dls.one_batch()[0].cpu()]

print(f"{'Method':<20} {'Accuracy':>10} {'vs Baseline':>12}")
print("-" * 45)
print(f"{'original':<20} {baseline*100:>9.1f}%")

for method in ['tucker', 'svd']:
    # Standard (no calibration)
    m_std = decomposer.decompose(copy.deepcopy(learn.model), 0.5, method=method)
    acc_std = Learner(dls, m_std, metrics=accuracy).validate()[1]
    
    # Activation-aware (with calibration data)
    m_aware = decomposer.decompose(copy.deepcopy(learn.model), 0.5, method=method, data=cal_data)
    acc_aware = Learner(dls, m_aware, metrics=accuracy).validate()[1]
    
    print(f"{method:<20} {acc_std*100:>9.1f}% {(acc_std-baseline)*100:>+11.1f}%")
    print(f"{method}+data{'':<11} {acc_aware*100:>9.1f}% {(acc_aware-baseline)*100:>+11.1f}%")

## 6. Auto Rank with `energy_threshold`

Instead of guessing `percent_removed`, let the decomposer pick the right rank automatically:

```python
# Keep 99% of singular value energy — minimal accuracy loss
Conv_Decomposer().decompose(model, energy_threshold=0.99)

# Keep 90% — more aggressive compression
Conv_Decomposer().decompose(model, energy_threshold=0.90)
```

`energy_threshold` and `percent_removed` are mutually exclusive. Higher threshold = less compression, better accuracy.

## 7. Combining with Other Techniques

Decomposition works well as a first step before other compressions:

```python
from fasterai.misc.all import Conv_Decomposer, BN_Folder

# 1. Fold BatchNorm
model = BN_Folder().fold(model)

# 2. Decompose (activation-aware Tucker)
model = Conv_Decomposer().decompose(model, 0.5, data=[cal_batch])

# 3. Fine-tune
learn = Learner(dls, model, metrics=accuracy)
learn.fit_one_cycle(3, 1e-4)

# 4. Quantize for deployment
from fasterai.quantize.quantizer import Quantizer
model = Quantizer(backend='torchao', method='int8_weight_only').quantize(model)
```

---

## Summary

| Method | Layers | What it decomposes | Best for |
|--------|--------|-------------------|----------|
| `'tucker'` | 3 | Both channel dims | General purpose (default) |
| `'svd'` | 2 | Output channels | Moderate compression, less overhead |
| `'spatial'` | 2 | Kernel K×K → K×1 + 1×K | Small kernels (3×3, 5×5) |
| `'cp'` | 4 | Channels + spatial | Maximum compression |

| Feature | Description |
|---------|-------------|
| `Conv_Decomposer().decompose(model, 0.5)` | Tucker decomposition (default) |
| `method='svd'\|'tucker'\|'spatial'\|'cp'` | Choose decomposition method |
| `energy_threshold=0.99` | Auto rank selection (keep 99% energy) |
| `layers=['layer1'], exclude=['conv1']` | Per-layer control |

---

## See Also

- [FC Decomposer](tutorial.fc_decomposer.html) - SVD decomposition for Linear layers
- [BN Folding](bn_folding.html) - Fold BatchNorm before decomposition
- [Pruner Tutorial](../prune/pruner.html) - Apply after decomposition for further compression